# Komodo-7B | SRL Krisis — Google Colab Pro (GPU + gdown)

**Notebook terpisah untuk Komodo-7B** berdasarkan `Checkpoint1_SRL_FALSE-EVENT.ipynb`. Melatih **Komodo-7B** saja, menggunakan **4-bit QLoRA**, split **SEED=42** dan penyaringan unlabeled yang mengikuti notebook utama.

## 1. Dependensi dan GPU


In [ ]:
import os, shutil, sys, subprocess
from pathlib import Path

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_HUB_CACHE"] = "/content/hf_cache/hub"
os.environ["HF_XET_CACHE"] = "/content/hf_cache/xet"
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "gdown>=5.2", "transformers>=4.49,<5", "accelerate>=1,<2",
    "peft>=0.15,<1", "bitsandbytes>=0.45", "huggingface_hub>=0.30,<2",
    "pandas", "numpy", "scikit-learn", "safetensors"
])

import torch
assert torch.cuda.is_available(), (
    "GPU CUDA TIDAK AKTIF. Colab: Runtime > Change runtime type > GPU, lalu jalankan ulang."
)
DEVICE = torch.device("cuda:0")
prop = torch.cuda.get_device_properties(0)
free_disk = shutil.disk_usage("/content").free / 1024**3
print("GPU     :", prop.name)
print("VRAM    :", round(prop.total_memory / 1024**3, 2), "GiB")
print("PyTorch :", torch.__version__, "| CUDA:", torch.version.cuda)
print("Disk kosong:", round(free_disk, 2), "GiB")
assert free_disk >= 20, (
    f"Disk Colab tinggal {free_disk:.1f} GiB. Sediakan >=20 GiB sebelum mengunduh Komodo-7B."
)
if prop.total_memory / 1024**3 < 16:
    print("PERINGATAN: VRAM <16 GiB berisiko OOM saat training QLoRA.")

GPU     : NVIDIA L4
VRAM    : 22.03 GiB
PyTorch : 2.11.0+cu130 | CUDA: 13.0
Disk kosong: 192.82 GiB


## 2. Konfigurasi eksperimen dan penyimpanan

Parameter utama mengikuti notebook lama: **10 epoch**, **learning rate 2e-5**, **effective batch 16**, **seed 42**, **test 250 tweet per kejadian**, **5.850 unlabeled**, threshold **0,9 / 0,8 / 0,7**. QLoRA membutuhkan *micro-batch 1* + *gradient accumulation 16* (berbeda dari batch fisik 16). Val 20% dari data latih berlabel dipakai untuk early stopping.

In [ ]:
import re, json, time, math, random, gc, glob, collections
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
from IPython.display import display

SEED = 42
EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
BATCH_SIZE = 16
MICRO_BATCH_SIZE = 1
GRAD_ACCUM = BATCH_SIZE // MICRO_BATCH_SIZE
MAX_LEN = 256
PATIENCE = 3
VAL_RATIO = 0.2
MAX_GRAD_NORM = 1.0
TEST_PER_EVENT = 250
N_UNLABELED = 5850
THRESHOLDS = [0.9, 0.8, 0.7]
MAX_ITER = 1
RESUME = True
KOMODO_MODEL = "Yellow-AI-NLP/komodo-7b-base"

from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = Path("/content/drive/MyDrive/SRL_Komodo_Colab/outputs")
for name in ["results", "models", "figures"]:
    (OUT_DIR / name).mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path("/content/srl_data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if BF16 else torch.float16
print("Output :", OUT_DIR)
print("Precision:", "bf16" if BF16 else "fp16")
print("Effective batch:", MICRO_BATCH_SIZE, "x", GRAD_ACCUM, "=", BATCH_SIZE)
print("MAX_ITER:", MAX_ITER)

Mounted at /content/drive
Output : /content/drive/MyDrive/SRL_Komodo_Colab/outputs
Precision: bf16
Effective batch: 1 x 16 = 16
MAX_ITER: 1


## 3. **Labeled Data** dan **Preprocessed Data**

In [ ]:
import gdown

LINK_LABELED = "https://drive.google.com/drive/folders/1b5XDuOCBUsmdEJU80kz26HwWRLlNRID1"
LINK_PRE = "https://drive.google.com/drive/folders/1CUtVt1Fd_1tQux1bBjwsdzeKcWbp1Mzm"


def find_one(filename, root=DATA_ROOT):
    files = sorted(p for p in root.rglob("*") if p.is_file() and p.name.lower() == filename.lower())
    if not files:
        return None
    if len(files) > 1:
        raise RuntimeError(f"Ada lebih dari satu file bernama {filename}: {files}. Periksa dataset.")
    return files[0]


def unduh_folder(link, target, contoh_file):
    target.mkdir(parents=True, exist_ok=True)
    if find_one(contoh_file, target) is not None:
        print(f"[SKIP] {contoh_file} sudah ada di {target}")
        return
    print(f"Mengunduh {target.name} ...")
    files = gdown.download_folder(url=link, output=str(target), quiet=False)
    if not files and find_one(contoh_file, target) is None:
        raise RuntimeError(
            f"gdown tidak mengunduh file yang diperlukan dari {link}. "
            "Pastikan akses folder DAN file bersifat publik (Anyone with the link)."
        )
    if find_one(contoh_file, target) is None:
        raise FileNotFoundError(f"Folder terunduh, tapi {contoh_file} tidak ditemukan pada {target}.")


unduh_folder(LINK_LABELED, DATA_ROOT / "labeled", "all-labeled-data.csv")
unduh_folder(LINK_PRE, DATA_ROOT / "preprocessed", "flood-preprocessed.csv")

LAB_CSV = find_one("all-labeled-data.csv", DATA_ROOT / "labeled")
EV_EN = {"BAN": "flood", "GEM": "earthquake", "KEB": "fire", "KEC": "accident"}
PRE_FILES = {ev: find_one(f"{ev}-preprocessed.csv", DATA_ROOT / "preprocessed") for ev in EV_EN.values()}
if any(p is None for p in PRE_FILES.values()):
    raise FileNotFoundError(f"File preprocessed belum lengkap: {PRE_FILES}")
print("Labeled CSV:", LAB_CSV)
print("Preprocessed:", PRE_FILES)

Mengunduh labeled ...


Retrieving folder contents


Processing file 1xeUOxpuCVsXEvg7L-FZ0bSXhBUmWWs74 all-labeled-data.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1xeUOxpuCVsXEvg7L-FZ0bSXhBUmWWs74
To: /content/srl_data/labeled/all-labeled-data.csv
100%|██████████| 3.83M/3.83M [00:00<00:00, 89.7MB/s]
Download completed
Retrieving folder contents


Mengunduh preprocessed ...
Processing file 1CqSfVABL0nHTU9Nl7EGPNh0J7-ZfJczW accident-preprocessed.csv
Processing file 1ijh1a8DWfE_K9oUYJEIkIn_pkWHwyliJ earthquake-preprocessed.csv
Processing file 1yKH8n6D7PxxQmQlFU_P_XsuCJFhUgQTn fire-preprocessed.csv
Processing file 1n2PrcoOys5i94aBSYNiJgnMVcUJ9UvVM flood-preprocessed.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1CqSfVABL0nHTU9Nl7EGPNh0J7-ZfJczW
To: /content/srl_data/preprocessed/accident-preprocessed.csv
100%|██████████| 1.57M/1.57M [00:00<00:00, 155MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ijh1a8DWfE_K9oUYJEIkIn_pkWHwyliJ
To: /content/srl_data/preprocessed/earthquake-preprocessed.csv
100%|██████████| 1.62M/1.62M [00:00<00:00, 109MB/s]
Downloading...
From: https://drive.google.com/uc?id=1yKH8n6D7PxxQmQlFU_P_XsuCJFhUgQTn
To: /content/srl_data/preprocessed/fire-preprocessed.csv
100%|██████████| 6.35M/6.35M [00:00<00:00, 53.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1n2PrcoOys5i94aBSYNiJgnMVcUJ9UvVM
To: /content/srl_data/preprocessed/flood-preprocessed.csv
100%|██████████| 1.55M/1.55M [00:00<00:00, 145MB/s]

Labeled CSV: /content/srl_data/labeled/all-labeled-data.csv
Preprocessed: {'flood': PosixPath('/content/srl_data/preprocessed/flood-preprocessed.csv'), 'earthquake': PosixPath('/content/srl_data/preprocessed/earthquake-preprocessed.csv'), 'fire': PosixPath('/content/srl_data/preprocessed/fire-preprocessed.csv'), 'accident': PosixPath('/content/srl_data/preprocessed/accident-preprocessed.csv')}



Download completed


## 4. Split data

Urutan `text_id` mengikuti `groupby(sort=False)` dari `all-labeled-data.csv`, kemudian `random.Random(42)` mengacak **per kejadian** dan mengambil 250 data uji per kejadian. Pemilihan 5.850 data tanpa label mengikuti normalisasi teks, deduplikasi, cosine similarity 0,8, dan shuffle di notebook asli.

In [ ]:
import hashlib

raw_df = pd.read_csv(LAB_CSV, keep_default_na=False)
expected_cols = {"text_id", "token", "argument", "entity"}
assert expected_cols.issubset(raw_df.columns), f"Kolom dataset tidak sesuai: {raw_df.columns.tolist()}"
df = raw_df[raw_df["token"].astype(str).str.strip() != ""].copy()
tweets = []
for tid, g in df.groupby("text_id", sort=False):
    tweets.append({"text_id": tid, "event": tid[:3], "tokens": g["token"].astype(str).tolist(),
                   "labels": g["argument"].tolist(), "entity": g["entity"].tolist()})
LABELS = ["O"] + sorted(l for l in df["argument"].unique() if l != "O")
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
assert len(LABELS) == 15, f"Label berbeda dari notebook utama: {LABELS}"

rng = random.Random(SEED)
test, train_labeled = [], []
for ev in sorted(EV_EN):
    group = [t for t in tweets if t["event"] == ev]
    rng.shuffle(group)
    test += group[:TEST_PER_EVENT]
    train_labeled += group[TEST_PER_EVENT:]
assert len(test) == 1000, f"Test harus 1000 tweet, aktual {len(test)}"
print("Tweet labeled:", len(tweets), "| latih:", len(train_labeled), "| test:", len(test))
print("LABELS:", LABELS)

pre_df = {ev: pd.read_csv(path, keep_default_na=False) for ev, path in PRE_FILES.items()}
for ev, d in pre_df.items():
    assert "full_text" in d.columns, f"Kolom full_text tidak ditemukan: {ev}"


def norm_text(s):
    return re.sub(r"[\W_]+", "", str(s).lower())


def simple_tokenize(text):
    return re.findall(r'[()\[\]?!.,;:"]|[^\s()\[\]?!.,;:"]+', str(text).lower())


texts = [" ".join(t["tokens"]) for t in tweets]
labeled_keys = {norm_text(x) for x in texts}
pool, matched = [], set()
for ev in EV_EN.values():
    for txt in pre_df[ev]["full_text"].astype(str):
        k = norm_text(txt)
        if k in labeled_keys:
            matched.add(k)
            continue
        toks = simple_tokenize(txt)
        if toks:
            pool.append({"tokens": toks, "key": k})

seen, uniq = set(), []
for u in pool:
    if u["key"] not in seen:
        seen.add(u["key"])
        uniq.append(u)

vec_all = CountVectorizer(token_pattern=r"\S+").fit(texts + [" ".join(u["tokens"]) for u in uniq])
L_mat = normalize(vec_all.transform(texts))
max_to_labeled = np.zeros(len(uniq))
for s0 in range(0, len(uniq), 2000):
    P = normalize(vec_all.transform([" ".join(u["tokens"]) for u in uniq[s0:s0 + 2000]]))
    max_to_labeled[s0:s0 + 2000] = (P @ L_mat.T).max(axis=1).toarray().ravel()
clean = [u for u, m in zip(uniq, max_to_labeled) if m < 0.8]
random.Random(SEED).shuffle(clean)
unlabeled = [{"tokens": u["tokens"]} for u in clean[:N_UNLABELED]]
assert len(unlabeled) == N_UNLABELED, f"Data unlabeled kurang: {len(unlabeled)}"
print("Unlabeled:", len(unlabeled), "| setelah filter:", len(clean))

manifest = {
    "seed": SEED,
    "labels": LABELS,
    "train_ids_sha256": hashlib.sha256(json.dumps([t["text_id"] for t in train_labeled]).encode()).hexdigest(),
    "test_ids_sha256": hashlib.sha256(json.dumps([t["text_id"] for t in test]).encode()).hexdigest(),
    "unlabeled_sha256": hashlib.sha256(json.dumps([u["tokens"] for u in unlabeled], ensure_ascii=False).encode()).hexdigest(),
}
manifest_path = OUT_DIR / "results" / "komodo_data_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print("Test SHA256:", manifest["test_ids_sha256"])
print("Manifest tersimpan:", manifest_path)

Tweet labeled: 4150 | latih: 3150 | test: 1000
LABELS: ['O', 'ACCIDENT-EVENT', 'AFFECTEDOBJECTS-ARG', 'DEATHVICTIM-ARG', 'EARTHQUAKE-EVENT', 'FALSE-EVENT', 'FIRE-EVENT', 'FLOOD-EVENT', 'INFORMATION-ARG', 'OFFICER-ARG', 'PLACE-ARG', 'REASON-ARG', 'STREET-ARG', 'TIME-ARG', 'WOUNDVICTIM-ARG']
Unlabeled: 5850 | setelah filter: 52157
Test SHA256: a5da6841c3e159070ae89740744f713611e5975854e4353438152e3c292f6171
Manifest tersimpan: /content/drive/MyDrive/SRL_Komodo_Colab/outputs/results/komodo_data_manifest.json


## 5. Autentikasi Hugging Face

In [ ]:
from google.colab import userdata
from huggingface_hub import hf_hub_download

try:
    HF_TOKEN1 = userdata.get("HF_TOKEN1")
except Exception as exc:
    raise RuntimeError(
        "Buat Secret bernama HF_TOKEN1 di sidebar Colab (ikon kunci), "
        "aktifkan Notebook access, kemudian jalankan ulang cell ini."
    ) from exc
if not HF_TOKEN1:
    raise RuntimeError("Secret HF_TOKEN1 kosong.")

config_path = hf_hub_download(repo_id=KOMODO_MODEL, filename="config.json", token=HF_TOKEN1)
print("Akses Komodo-7B berhasil. Config ditemukan:", config_path)

config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

Akses Komodo-7B berhasil. Config ditemukan: /content/hf_cache/hub/models--Yellow-AI-NLP--komodo-7b-base/snapshots/bf2ca08ef273698557ef5cc6a3524b8a71934081/config.json


## 6. Metrik dan format hasil **sama dengan notebook utama**

Nama key: `Komodo-7B | tanpa self-training` dan `Komodo-7B | self-training θ=0.9/0.8/0.7`. Field JSON `key`, `family`, `metrics`, `pred`, `conf`, `hist`, `train_time_s`, `infer_s_per_1000`, `peak_mem_gb`, `device`, `threshold`, `st_log`, `params`. File JSON disimpan lewat nama `slug(key)` yang digunakan di notebook utama.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def spans(labels):
    out, start, cur = [], None, None
    for i, lab in enumerate(list(labels) + ["O"]):
        if lab != cur:
            if cur not in (None, "O"):
                out.append((start, i, cur))
            start, cur = i, lab
    return out


def evaluate(gold, pred):
    assert len(gold) == len(pred)
    assert all(len(g) == len(p) for g, p in zip(gold, pred)), "Panjang prediksi != label gold"
    g, p = sum(gold, []), sum(pred, [])
    non_o = [l for l in LABELS if l != "O"]
    acc = accuracy_score(g, p)
    P, R, F, _ = precision_recall_fscore_support(g, p, labels=LABELS, average="weighted", zero_division=0)
    F_wn = precision_recall_fscore_support(g, p, labels=non_o, average="weighted", zero_division=0)[2]
    F_mn = precision_recall_fscore_support(g, p, labels=non_o, average="macro", zero_division=0)[2]
    f_lab = precision_recall_fscore_support(g, p, labels=LABELS, average=None, zero_division=0)[2]
    per_label = {l: float(v) for l, v in zip(LABELS, f_lab)}
    gs = {(i,) + s for i, seq in enumerate(gold) for s in spans(seq)}
    ps = {(i,) + s for i, seq in enumerate(pred) for s in spans(seq)}
    tp = len(gs & ps)
    sp = tp / len(ps) if ps else 0.0
    sr = tp / len(gs) if gs else 0.0
    sf = 2 * sp * sr / (sp + sr) if sp + sr else 0.0
    return {"accuracy": acc, "precision_w": P, "recall_w": R, "f1_w": F,
            "f1_w_noO": F_wn, "f1_macro_noO": F_mn, "f1_span": sf,
            "f1_FALSE": per_label["FALSE-EVENT"], "f1_FLOOD": per_label["FLOOD-EVENT"], "f1_per_label": per_label}


def slug(s):
    return re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")


def res_path(key):
    return OUT_DIR / "results" / (slug(key) + ".json")


def save_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    os.replace(tmp, path)


def load_results():
    result = {}
    for p in (OUT_DIR / "results").glob("Komodo_7B_*.json"):
        with open(p, encoding="utf-8") as f:
            r = json.load(f)
        if r.get("family") == "Komodo-7B" and "metrics" in r:
            result[r["key"]] = r
    return result


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()


def peak_gb():
    return round(torch.cuda.max_memory_allocated() / 1e9, 2)


def record(key, model, predict_fn, hist, train_s, threshold=None, st_log=None):
    test_in = [{"tokens": t["tokens"]} for t in test]
    t0 = time.time()
    pred, conf = predict_fn(model, test_in)
    torch.cuda.synchronize()
    inference_s = time.time() - t0
    metrics = evaluate([t["labels"] for t in test], pred)
    out = {
        "key": key, "family": "Komodo-7B", "metrics": metrics,
        "pred": pred, "conf": conf, "hist": hist,
        "train_time_s": train_s, "infer_s_per_1000": 1000 * inference_s / len(test),
        "peak_mem_gb": peak_gb(), "device": "cuda",
        "params": int(sum(p.numel() for p in model.parameters())),
        "data_manifest": manifest["test_ids_sha256"],
    }
    if threshold is not None:
        out.update({"threshold": threshold, "st_log": st_log or []})
    save_json(res_path(key), out)
    print(f"[SAVED] {key} | F1_w={metrics['f1_w']:.4f} | {res_path(key)}")
    return out

## 7. Tokenizer dan data loader untuk Komodo (token classification)

Penyelarasan label mengikuti notebook utama: hanya subword pertama untuk tiap kata memperoleh label, sisanya `-100`. Pad token menggunakan EOS karena tokenizer Llama sering tidak memiliki pad token. Kata yang terpotong oleh `MAX_LEN` tetap diprediksi `O` pada hasil seperti notebook utama; hasil ini **bukan** cara ideal untuk mengukur performa label terpotong, tetapi menjaga kompatibilitas.

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training, PeftModel
from torch.utils.data import DataLoader
from contextlib import nullcontext

TOKENIZER = AutoTokenizer.from_pretrained(KOMODO_MODEL, token=HF_TOKEN1, use_fast=True)
assert TOKENIZER.is_fast, "Tokenizer harus fast agar .word_ids() tersedia."
TOKENIZER.padding_side = "right"
if TOKENIZER.pad_token is None:
    TOKENIZER.pad_token = TOKENIZER.eos_token
assert TOKENIZER.pad_token_id is not None


def collate_tf(batch):
    enc = TOKENIZER([b["tokens"] for b in batch], is_split_into_words=True,
                    truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
    length = enc["input_ids"].shape[1]
    labels = torch.full((len(batch), length), -100, dtype=torch.long)
    first_idx = torch.full((len(batch), length), -1, dtype=torch.long)
    for i, b in enumerate(batch):
        prev = None
        for j, w in enumerate(enc.word_ids(batch_index=i)):
            if w is not None and w != prev:
                first_idx[i, j] = w
                if b.get("labels") is not None:
                    labels[i, j] = label2id[b["labels"][w]]
            prev = w
    return {"inputs": {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]},
            "labels": labels, "first_idx": first_idx,
            "n_words": [len(b["tokens"]) for b in batch]}

batch_demo = collate_tf([train_labeled[0]])
print("Demo input:", tuple(batch_demo["inputs"]["input_ids"].shape))
print("Subword pertama berlabel:", int((batch_demo["labels"] != -100).sum()))
print("Token ID padding:", TOKENIZER.pad_token_id)

Demo input: (1, 35)
Subword pertama berlabel: 18
Token ID padding: 2


## 8. Komodo-7B 4-bit + LoRA dan classifier head

Classifier head (`score`) diinisialisasi untuk 15 label. Ini **normal** karena Komodo awalnya causal language model. `modules_to_save=['score']` memastikan head turut dilatih dan disimpan ke adapter checkpoint. GPU tetap digunakan untuk bobot model, dengan quantization 4-bit NF4 agar kebutuhan VRAM lebih hemat.

In [ ]:
QCONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)


def base_model():
    set_seed()
    model = AutoModelForTokenClassification.from_pretrained(
        KOMODO_MODEL,
        num_labels=len(LABELS),
        id2label=id2label,
        label2id=label2id,
        token=HF_TOKEN1,
        quantization_config=QCONFIG,
        torch_dtype=COMPUTE_DTYPE,
        device_map={"": 0},
        low_cpu_mem_usage=True,
    )
    model.config.pad_token_id = TOKENIZER.pad_token_id
    model.config.use_cache = False
    return model


def build_model(init_adapter=None):
    model = base_model()
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    if init_adapter is None:
        lcfg = LoraConfig(
            task_type=TaskType.TOKEN_CLS,
            r=16, lora_alpha=32, lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            modules_to_save=["score"],
            bias="none",
        )
        model = get_peft_model(model, lcfg)
    else:
        model = PeftModel.from_pretrained(model, str(init_adapter), is_trainable=True)
    assert any("score" in n and p.requires_grad for n, p in model.named_parameters()), (
        "Classifier score tidak trainable. Cek modules_to_save dan versi PEFT."
    )
    model.print_trainable_parameters()
    return model


def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=COMPUTE_DTYPE)


def split_train_val(items):
    idx = list(range(len(items)))
    random.Random(SEED).shuffle(idx)
    n_val = int(len(items) * VAL_RATIO)
    return [items[i] for i in idx[n_val:]], [items[i] for i in idx[:n_val]]

## 9. Training QLoRA — grad accumulation, early stopping, dan evaluasi

Model awal dan model tiap threshold disimpan terpisah. Selama fitting, *best state* dipilih dari loss validasi (**tidak test**). Jika runtime Colab putus, output yang **sudah selesai dan punya JSON + checkpoint** akan dilewati saat cell training dijalankan ulang. Epoch yang terputus tidak otomatis bisa di-resume dari langkah terakhir.

In [ ]:
import torch.nn as nn


def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in names}


def param_groups(model, wd):
    decay, no_decay = [], []
    for module in model.modules():
        is_norm = isinstance(module, nn.LayerNorm) or "norm" in type(module).__name__.lower()
        for n, p in module.named_parameters(recurse=False):
            if p.requires_grad:
                (no_decay if is_norm or "bias" in n else decay).append(p)
    return [{"params": decay, "weight_decay": wd}, {"params": no_decay, "weight_decay": 0.0}]


@torch.no_grad()
def eval_loss(model, items):
    model.eval()
    ce = nn.CrossEntropyLoss(ignore_index=-100)
    total, n = 0.0, 0
    for batch in DataLoader(items, batch_size=MICRO_BATCH_SIZE, collate_fn=collate_tf, shuffle=False):
        inp = {k: v.to(DEVICE) for k, v in batch["inputs"].items()}
        lab = batch["labels"].to(DEVICE)
        with autocast_ctx():
            logits = model(**inp).logits
            loss = ce(logits.float().reshape(-1, logits.shape[-1]), lab.reshape(-1))
        total += loss.item() * len(lab)
        n += len(lab)
    return total / max(n, 1)


def fit_komodo(model, items, desc="Komodo"):
    tr, va = split_train_val(items)
    generator = torch.Generator().manual_seed(SEED)
    dl = DataLoader(tr, batch_size=MICRO_BATCH_SIZE, shuffle=True, collate_fn=collate_tf, generator=generator)
    optim = torch.optim.AdamW(param_groups(model, WEIGHT_DECAY), lr=LEARNING_RATE)
    steps_per_epoch = math.ceil(len(dl) / GRAD_ACCUM)
    total_steps = max(1, steps_per_epoch * EPOCHS)
    sched = torch.optim.lr_scheduler.LambdaLR(optim, lambda step: max(0.0, 1 - step / total_steps))
    scaler = torch.amp.GradScaler("cuda", enabled=not BF16)
    ce = nn.CrossEntropyLoss(ignore_index=-100)
    params = [p for p in model.parameters() if p.requires_grad]
    best_loss, best_state, bad, hist = float("inf"), None, 0, []

    for ep in range(1, EPOCHS + 1):
        model.train()
        optim.zero_grad(set_to_none=True)
        t0, sum_loss, n = time.time(), 0.0, 0
        for i, batch in enumerate(dl):
            inp = {k: v.to(DEVICE) for k, v in batch["inputs"].items()}
            lab = batch["labels"].to(DEVICE)
            with autocast_ctx():
                logits = model(**inp).logits
                loss = ce(logits.float().reshape(-1, logits.shape[-1]), lab.reshape(-1))
            group_start = (i // GRAD_ACCUM) * GRAD_ACCUM
            n_group = min(GRAD_ACCUM, len(dl) - group_start)
            scaler.scale(loss / n_group).backward()
            if (i + 1) % GRAD_ACCUM == 0 or (i + 1) == len(dl):
                scaler.unscale_(optim)
                nn.utils.clip_grad_norm_(params, MAX_GRAD_NORM)
                scaler.step(optim)
                scaler.update()
                sched.step()
                optim.zero_grad(set_to_none=True)
            sum_loss += loss.detach().item() * len(lab)
            n += len(lab)
            del logits, loss
        val = eval_loss(model, va)
        row = {"epoch": ep, "train_loss": sum_loss / max(n, 1),
               "val_loss": val, "detik": time.time() - t0}
        hist.append(row)
        print(f"[{desc}] epoch {ep:2d} | train={row['train_loss']:.4f} | val={val:.4f} | {row['detik']:.0f}s")
        if val < best_loss - 1e-4:
            best_loss, bad = val, 0
            best_state = trainable_state(model)
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"[{desc}] early stopping pada epoch {ep}")
                break
    if best_state is not None:
        model.load_state_dict(best_state, strict=False)
    return model, hist


@torch.no_grad()
def predict_komodo(model, items):
    model.eval()
    preds, confs = [], []
    for batch in DataLoader(items, batch_size=MICRO_BATCH_SIZE, shuffle=False, collate_fn=collate_tf):
        inp = {k: v.to(DEVICE) for k, v in batch["inputs"].items()}
        with autocast_ctx():
            prob = model(**inp).logits.float().softmax(-1).cpu()
        first_idx = batch["first_idx"]
        for i, n in enumerate(batch["n_words"]):
            lab, cf = ["O"] * n, [None] * n
            positions = (first_idx[i] >= 0).nonzero().squeeze(-1)
            if positions.numel():
                scores, labels = prob[i, positions].max(-1)
                for word, score, label in zip(first_idx[i, positions].tolist(), scores.tolist(), labels.tolist()):
                    lab[word], cf[word] = id2label[label], score
            preds.append(lab)
            confs.append([x for x in cf if x is not None])
    return preds, confs

## 10. Eksperimen Komodo-7B (awal dan 3 threshold)

- Model awal disimpan sebagai adapter di `models/Komodo_7B_init/` dan JSON di `results/Komodo_7B_tanpa_self_training.json`.
- Setiap threshold dimulai ulang dari adapter model awal yang sama, bukan dari threshold sebelumnya.
- Pseudo-label diterima berdasarkan **rata-rata confidence per tweet**; data latih ditambah hasil pseudo-label, lalu model dilatih ulang.

In [ ]:
def train_initial():
    key = "Komodo-7B | tanpa self-training"
    adapter_dir = OUT_DIR / "models" / "Komodo_7B_init"
    if RESUME and res_path(key).exists() and (adapter_dir / "adapter_config.json").exists():
        print("[RESUME] Model awal sudah ada:", res_path(key))
        return adapter_dir
    set_seed()
    clear_gpu()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    model = build_model()
    model, hist = fit_komodo(model, train_labeled, desc="Komodo awal")
    model.save_pretrained(adapter_dir, safe_serialization=True)
    TOKENIZER.save_pretrained(adapter_dir)
    record(key, model, predict_komodo, hist, time.time() - t0)
    del model
    clear_gpu()
    return adapter_dir


def run_threshold(T, adapter_dir):
    key = f"Komodo-7B | self-training θ={T}"
    adapter_out = OUT_DIR / "models" / slug(key)
    if RESUME and res_path(key).exists() and (adapter_out / "adapter_config.json").exists():
        print("[RESUME] Melewati:", key)
        return

    set_seed()
    clear_gpu()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    model = build_model(init_adapter=adapter_dir)
    L = list(train_labeled)
    U = [{"tokens": u["tokens"]} for u in unlabeled]
    st_log, it = [], 0
    while U and (MAX_ITER is None or it < MAX_ITER):
        it += 1
        preds, confs = predict_komodo(model, U)
        selected = [i for i, c in enumerate(confs) if len(c) and float(np.mean(c)) >= T]
        entry = {"iter": it, "diterima": len(selected), "sisa_U": len(U) - len(selected)}
        print(f"[θ={T}] iterasi {it}: {len(selected)} dari {len(U)} lolos")
        if not selected:
            st_log.append(entry)
            break
        picked = set(selected)
        L += [{"tokens": U[i]["tokens"], "labels": preds[i]} for i in selected]
        U = [u for i, u in enumerate(U) if i not in picked]
        del preds, confs
        clear_gpu()
        model, hist = fit_komodo(model, L, desc=f"Komodo θ={T} iterasi {it}")
        entry.update({"hist": hist, "ukuran_L": len(L)})
        st_log.append(entry)

    model.save_pretrained(adapter_out, safe_serialization=True)
    TOKENIZER.save_pretrained(adapter_out)
    record(key, model, predict_komodo, None, time.time() - t0, threshold=T, st_log=st_log)
    del model
    clear_gpu()


INIT_ADAPTER_DIR = train_initial()
for threshold in THRESHOLDS:
    run_threshold(threshold, INIT_ADAPTER_DIR)

print("\nSEMUA eksperimen Komodo yang dipilih selesai.")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

model-00006-of-00006.safetensors:   0%|          | 0.00/2.73G [00:00<?, ?B/s]

model-00003-of-00006.safetensors:   0%|          | 0.00/4.86G [00:00<?, ?B/s]

model-00002-of-00006.safetensors:   0%|          | 0.00/4.86G [00:00<?, ?B/s]

... (27 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


[Komodo θ=0.7 iterasi 1] epoch  1 | train=0.2100 | val=0.1898 | 3208s
[Komodo θ=0.7 iterasi 1] epoch  2 | train=0.1830 | val=0.1882 | 3241s
[Komodo θ=0.7 iterasi 1] epoch  3 | train=0.1602 | val=0.1913 | 3241s
[Komodo θ=0.7 iterasi 1] epoch  4 | train=0.1445 | val=0.2008 | 3255s
[Komodo θ=0.7 iterasi 1] epoch  5 | train=0.1293 | val=0.1933 | 3271s
[Komodo θ=0.7 iterasi 1] early stopping pada epoch 5


/usr/local/lib/python3.13/dist-packages/peft/utils/other.py:1508: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-6ae40a8c-1e87c1a6194330ff6db3cec4;664c346c-ce49-4de9-9e76-f1c03af0d90f)

Cannot access gated repo for url https://huggingface.co/Yellow-AI-NLP/komodo-7b-base/resolve/main/config.json.
Access to model Yellow-AI-NLP/komodo-7b-base is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in Yellow-AI-NLP/komodo-7b-base.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/utils/save_and_load.py:313: UserWarning: Could not find a config file in Yellow-AI-NLP/komodo-7b-base - will assume that the vocabulary was not modified.
  warnings.warn(


[SAVED] Komodo-7B | self-training θ=0.7 | F1_w=0.8702 | /content/drive/MyDrive/SRL_Komodo_Colab/outputs/results/Komodo_7B_self_training_0_7.json

SEMUA eksperimen Komodo yang dipilih selesai.


## 11. Tabel hasil Komodo

In [ ]:
RES_KOMODO = load_results()
rows = []
for key, r in RES_KOMODO.items():
    m = r["metrics"]
    rows.append({"model": key, "F1 weighted": m["f1_w"], "Macro-F1 no O": m["f1_macro_noO"],
                 "F1 FALSE-EVENT": m["f1_FALSE"], "F1 span": m["f1_span"],
                 "accuracy": m["accuracy"], "waktu training menit": r["train_time_s"] / 60})
if rows:
    display(pd.DataFrame(rows).sort_values("F1 weighted", ascending=False))
else:
    print("Belum ada hasil Komodo yang selesai.")
for k, r in RES_KOMODO.items():
    assert len(r["pred"]) == len(test), f"Jumlah pred tidak sesuai: {k}"
    assert all(len(p) == len(t["tokens"]) for p, t in zip(r["pred"], test)), f"Panjang pred berbeda: {k}"
print("JSON valid untuk", len(RES_KOMODO), "eksperimen.")

,model,F1 weighted,Macro-F1 no O,F1 FALSE-EVENT,F1 span,accuracy,waktu training menit
2,Komodo-7B | self-training θ=0.8,0.871054,0.644627,0.818182,0.606308,0.878021,263.028844
3,Komodo-7B | self-training θ=0.7,0.870167,0.647432,0.821696,0.604588,0.876410,281.518494
1,Komodo-7B | self-training θ=0.9,0.869704,0.638300,0.807175,0.597316,0.877153,191.739226
0,Komodo-7B | tanpa self-training,0.868132,0.640002,0.810654,0.589471,0.875046,131.263732


JSON valid untuk 4 eksperimen.


## 12. Ekspor hasil JSON

File ZIP berisi `results/Komodo_7B_*.json` dan `komodo_data_manifest.json`.


In [ ]:
import zipfile
from google.colab import files

zip_path = Path("/content/komodo_results_untuk_notebook_lokal.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted((OUT_DIR / "results").glob("Komodo_7B_*.json")):
        z.write(p, arcname=f"results/{p.name}")
    z.write(manifest_path, arcname="komodo_data_manifest.json")
print("Arsip:", zip_path)
print("Hasil yang diekspor:", len(list((OUT_DIR / 'results').glob('Komodo_7B_*.json'))))
files.download(str(zip_path))